# LMECA2300: Advanced Numerical Methods
## Assignment 3

**Students:**
- Student 1: Victor Lepère (61502000)
- Student 2: Max Opdecam(77062000)

### Question 1: Pressure field on the segment

Given a segment made of two points, $\mathbf{r_1}'$ and $\mathbf{r_2}'$, and is denoted by $\Gamma'$, we recall that the complex amplitude of the pressure wave at $\mathbf r$ is given by
$$
p(\mathbf{r}) = \frac{\omega \rho_0}{4} \int_{\Gamma'} H_0^{(2)} (k \rho) v_n d\Gamma'
$$
Where
- $\omega$ is the angular frequency;
- $k$ is the wavenumber;
- $\rho$ is the distance between $\mathbf{r}$ and $\mathbf{r}'$;
- $v_n$ is the normal velocity, assumed constant in this context.

Here we assume that $\mathbf r$ lies on the segment $\Gamma'$. Hence we can split the segment into two smaller ones, from $\mathbf r_1'$ to $\mathbf r$ and from $\mathbf r$ to $\mathbf r_2'$, respectively denoted by $\Gamma_1'$ and $\Gamma_2'$. Therefore,
\begin{align*}
p(\mathbf{r}) &= \frac{\omega \rho_0}{4} \int_{\mathbf{r_1}'}^{\mathbf{r_2}'} H_0^{(2)} (k \rho(\mathbf{r}')) v_n d\mathbf{r}' \\
&= v_n \frac{\omega \rho_0}{4} \left [ \int_{\mathbf{r_1}'}^{\mathbf{r}} H_0^{(2)} (k \rho(\mathbf{r}')) d\mathbf{r}' + \int_{\mathbf{r}}^{\mathbf{r_2}'} H_0^{(2)} (k \rho(\mathbf{r}')) d\mathbf{r}' \right ] \\
&= v_n \frac{\omega \rho_0}{4} \left [ \int_{0}^{1} H_0^{(2)} (k ||\mathbf r - (\mathbf r_1' + s (\mathbf r - \mathbf r_1'))||_2 ) L_1 ds + \int_{0}^{1} H_0^{(2)} (k ||\mathbf r - (\mathbf r + s (\mathbf r_2' - \mathbf r))||_2 ) L_2 ds \right ] \\
\end{align*}

where $L_i = ||\mathbf r - \mathbf r_i'||_2$. We can now exploit the symmetry of the distance metric. Let $\bar s = \frac{\min(L_1, L_2)}{\max(L_1, L_2)}$. If $L_1 < L_2$, we have
\begin{align*}
p(\mathbf{r}) &= v_n \frac{\omega \rho_0}{4} \left [ (L_1 + L_2) \int_{0}^{\bar s} H_0^{(2)} (k \rho(s, \mathbf r_2') ) ds + L_2 \int_{\bar s}^1 H_0^{(2)} (k \rho(s, \mathbf r_2') ) ds \right ] \\ 
\end{align*}
Conversely, if $L_1 > L_2$, the integral becomes
\begin{align*}
p(\mathbf{r}) &= v_n \frac{\omega \rho_0}{4} \left [ L_1 \int_{0}^{1-\bar s} H_0^{(2)} (k \rho(s, \mathbf r_1') ) ds + (L_1 + L_2) \int_{1-\bar s}^1 H_0^{(2)} (k \rho(s, \mathbf r_1') ) ds \right ] \\ 
\end{align*}

For the first case, we can now analytically extract the singularities as follows

\begin{align*}
\int_{a}^{b} H_0^{(2)} (k \rho(s, \mathbf r_1') ) ds = \int_{a}^{b} H_0^{(2)} (k \rho(s, \mathbf r_1') ) + j \frac{2}{\pi} \ln (k \rho(s, \mathbf r_1') )  ds - j \frac{2}{\pi} \int_{a}^{b} \ln (k \rho(s, \mathbf r_1') )  ds
\end{align*}

We can compute the first integral using the composite Trapezium rule with integration nodes $a + \frac{i}{m} (b-a)$ where $i = 0, 1, \dots, m$ and by considering $1 - j \frac{2}{\pi} (\gamma - \ln 2)$ when $k \rho(s, \mathbf r_1') \in \mathcal{O} \left ( \frac{\lambda}{1000} \right )$. The second integral is calculated analytically as follows:
\begin{align*}
\int_{a}^{b} \ln (k \rho(s, \mathbf r_1') )  ds &= (b-a) \ln(k) +  \int_{a}^{b} \ln (\rho(s, \mathbf r_1') )  ds \\
\int_{a}^{b} \ln (\rho(s, \mathbf r_1') )  ds &= \int_{a}^{b} \ln ||\mathbf r - (\mathbf r_1' + s (\mathbf r - \mathbf r_1'))||_2  ds \\
&= \int_{a}^{b} \ln (1 - s) + \ln ||(\mathbf r - \mathbf r_1')||_2  ds \\
&= (b - a) \ln ||(\mathbf r - \mathbf r_1')||_2 + \left [ t (\ln t - 1) \right ]_{1-b}^{1-a}
\end{align*}
While for the other case it comes
\begin{align*}
\int_{a}^{b} \ln (\rho(s, \mathbf r_2') )  ds &= \int_{a}^{b} \ln ||\mathbf r - (\mathbf r + s (\mathbf r_2' - \mathbf r))||_2  ds \\
&= (b - a) \ln ||\mathbf r_2' - \mathbf r||_2 + \int_a^b \ln s  \ ds \\
&= (b - a) \ln ||\mathbf r_2' - \mathbf r||_2 + \left [ s (\ln s - 1) \right ]_a^b \\
\end{align*}

Find below the implementation of the above development.

In [ ]:
### Imports
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation
from scipy.special import hankel2
from math import ceil

### Physical constants
rho0 = 1.2 # air, 20°C, 1 atm
f = 1/16 # [hz]
w = 2*np.pi*f # air -> 343 = w/k
k = w/343
lambda_ = (2*np.pi)/k

### A segment
class Edge:
    def __init__(self,r1,r2):
        self.r1 = r1
        self.r2 = r2

    def __integrate_hankel(self, a, b, m, case):
        """
        Aux. function that integrates Hankel function from a to b

        @param a: lower intergration bound
        @param b: upper integration bound
        @param case: symmetric/general case
        @param m: number of integration points
        """
        s = np.linspace(a, b, m)
        if case == 0:
            diff = self.obs - (self.r1 + s[:, np.newaxis] * (self.obs - self.r1))
        elif case == 1:
            diff = s[:, np.newaxis] * (self.r2 - self.obs)
        else:
            diff = self.obs - (self.r1 + s[:, np.newaxis] * (self.r2 - self.r1))
        rho = np.linalg.norm(diff, axis=1)
        singularity_mask = k*rho <= lambda_/1000

        evals = np.zeros_like(rho, dtype=np.complex128)
        cst = 1 - 1j*(2/np.pi)*(np.euler_gamma - np.log(2))
        evals[singularity_mask] = cst
        subset_rho = rho[~singularity_mask]
        evals[~singularity_mask] = hankel2(0, k*subset_rho) + 1j*(2/np.pi)*np.log(k*subset_rho)

        h = (b - a) / m
        evals[0] *= 1/2
        evals[-1] *= 1/2
        first_term = h*np.sum(evals)

        if case == 0:
            if b == 1:
                last = 0
            else:
                last = (1-b)*(np.log(1-b)-1)
            second_term = -1j*(2/np.pi)*((b-a)*(np.log(k)+np.log(np.linalg.norm(self.obs-self.r1))) + ((1-a)*(np.log(1-a)-1) - last))
        elif case == 1:
            if a == 0:
                last = 0
            else:
                last = a*(np.log(a)-1)
            second_term = -1j*(2/np.pi)*((b-a)*(np.log(k)+np.log(np.linalg.norm(self.r2-self.obs))) + (b*(np.log(b)-1) - last))
        else:
            pass

        return first_term + second_term
    
    def integrate_sym(self, Vn, obs):
        """
        Integrates complex amplitude for an obsever point ON the segment, and thus takes advantage of symmetry.

        @param Vn: constant normal velocity
        @param obs: coord. of observer
        """
        self.obs = obs
        L1 = np.linalg.norm(obs - self.r1)
        L2 = np.linalg.norm(obs - self.r2)

        sbar = np.fmin(L1, L2) / np.fmax(L1, L2)

        m = 20
        if L1 == 0:
            return (Vn*w*rho0*L2/4)*self.__integrate_hankel(0, 1, m, 1)
        elif L2 == 0:
            return (Vn*w*rho0*L1/4)*self.__integrate_hankel(0, 1, m, 0)
        elif L1 > L2:
            return (Vn*w*rho0/4)*(2*L2*self.__integrate_hankel(0, 1-sbar, m, 0) + L2*self.__integrate_hankel(1-sbar, 1, m, 0))
        elif L1 < L2:
            return (Vn*w*rho0/4)*(L1*self.__integrate_hankel(0, sbar, m, 1) + 2*L1*self.__integrate_hankel(sbar, 1, m, 1))
        else:
            return (Vn*w*rho0*L1/2)*self.__integrate_hankel(0, 1, m, 0)

In [7]:
r1 = np.array([-lambda_/20, 0])
r2 = np.array([lambda_/20, 0])
# r1 = np.array([1, 0])
# r2 = np.array([1+lambda_/10, 0])
edge = Edge(r1, r2)

pressure_nodes = r1 + np.linspace(0, 1, 50)[:, np.newaxis] * (r2 - r1)
phasors = [edge.integrate_sym(1, obs) for obs in pressure_nodes]
phasors = np.array(phasors, dtype=np.complex128)

np.savetxt("phasors100.txt", phasors)

plt.rcParams["animation.html"] = "jshtml"
plt.ioff()

fig, ax = plt.subplots()

def animate(t):
    plt.cla()
    pressures = np.real(phasors * np.exp(1j*w*t))
    plt.plot(pressure_nodes[:, 0], pressures, linewidth=2)
    plt.plot([r1[0], r2[0]], [r1[1], r2[1]], 'k', linewidth=2)

    plt.xlim(r1[0] - 10, r2[0] + 10)
    plt.ylim(-150, 150)
    ax.set_title("Pressure field")
    fig.tight_layout()
    
matplotlib.animation.FuncAnimation(fig, animate, frames=100, interval=60)